# Dashboard station map — status join

Explore how the Streamlit **Station map** tab colors stations from bronze
lat/lon plus Layer 1 incidents and fusion trust alerts.

```bash
python -m pipelines quality
python -m pipelines detect
python -m pipelines fuse
streamlit run dashboard/app.py
```

Status priority: `escalated` > `quarantined` > `quality_only` > `monitored`.
Quarantined stations are always visible — never hidden.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path("..").resolve()
BRONZE_ROOT = ROOT / "data" / "bronze"
GOLD_ROOT = ROOT / "data" / "gold"

from dashboard.data import (
    STATUS_COLORS,
    STATUS_PRIORITY,
    available_dates,
    build_station_status,
    load_dashboard_frames,
    load_stations,
)

stations = load_stations(BRONZE_ROOT)
frames = load_dashboard_frames(GOLD_ROOT)
incidents = frames["incidents"]
fused = frames["trust_alerts"]
dates = available_dates(frames["metrics"], incidents, fused)
as_of = dates[-1] if dates else None

print(f"Stations: {len(stations)}")
print(f"L1 incidents: {len(incidents)}")
print(f"Fused alerts: {len(fused)}")
print(f"As-of date: {as_of}")
stations.head()

## 1. Station registry (lat / lon from conformed bronze)

In [ ]:
if stations.empty:
    print("No stations — ingest bronze so read_conformed can supply lat/lon.")
else:
    display(stations)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(stations["lon"], stations["lat"], s=80, c="#2a9d8f")
    for _, row in stations.iterrows():
        ax.annotate(row["location_name"][:20], (row["lon"], row["lat"]), fontsize=8)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    ax.set_title("Monitored stations")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 2. Build map status for as-of date

In [ ]:
status = build_station_status(stations, incidents, fused, as_of_date=as_of)
if status.empty:
    print("No station status rows.")
else:
    display(
        status[
            [
                "location_id",
                "location_name",
                "status",
                "date_local",
                "trust_score",
                "max_severity",
                "incident_rule_ids",
            ]
        ]
    )
    display(status["status"].value_counts().to_frame("stations"))

## 3. Color-coded scatter (same palette as the dashboard)

In [ ]:
HEX = {
    "escalated": "#e63946",
    "quarantined": "#f4a261",
    "quality_only": "#7b61ff",
    "monitored": "#2a9d8f",
}

if status.empty:
    print("Skip — no status frame.")
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, group in status.groupby("status"):
        ax.scatter(
            group["lon"],
            group["lat"],
            s=120,
            c=HEX.get(name, "#888"),
            label=name,
            edgecolors="white",
            linewidths=0.8,
        )
        for _, row in group.iterrows():
            ax.annotate(str(row["location_id"]), (row["lon"], row["lat"]), fontsize=8)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    ax.set_title(f"Station map status — {as_of or 'no date'}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("RGB palette (dashboard.pydeck):", STATUS_COLORS)

## 4. As-of date sweep (how status changes across gold days)

In [ ]:
if not dates:
    print("No gold dates — run quality (and fuse when alerts exist).")
else:
    rows = []
    for day in dates:
        day_status = build_station_status(stations, incidents, fused, as_of_date=day)
        counts = day_status["status"].value_counts().to_dict() if not day_status.empty else {}
        rows.append({"date_local": day, **{k: counts.get(k, 0) for k in STATUS_PRIORITY}})
    sweep = pd.DataFrame(rows).set_index("date_local")
    display(sweep)
    if len(sweep) > 0:
        sweep.plot(kind="bar", stacked=True, figsize=(8, 4), color=[HEX[k] for k in STATUS_PRIORITY])
        plt.ylabel("Stations")
        plt.title("Status mix by as-of date")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

## 5. Synthetic status demo (all four colors)

Useful when fusion gold is empty — mirrors `tests/test_dashboard.py`.

In [ ]:
demo_stations = pd.DataFrame(
    [
        {"location_id": 1, "location_name": "Escalated demo", "lat": -33.85, "lon": 151.21},
        {"location_id": 2, "location_name": "Quarantined demo", "lat": -34.03, "lon": 151.12},
        {"location_id": 3, "location_name": "Quality-only demo", "lat": -33.86, "lon": 151.17},
        {"location_id": 4, "location_name": "Monitored demo", "lat": -33.66, "lon": 151.31},
    ]
)
demo_incidents = pd.DataFrame(
    [
        {"location_id": 2, "date_local": "2026-01-08", "rule_id": "R2", "severity": "high"},
        {"location_id": 3, "date_local": "2026-01-08", "rule_id": "R4", "severity": "medium"},
    ]
)
demo_trust = pd.DataFrame(
    [
        {"location_id": 1, "date_local": "2026-01-08", "status": "escalated", "trust_score": 0.9},
        {"location_id": 2, "date_local": "2026-01-08", "status": "quarantined", "trust_score": 0.3},
    ]
)
demo = build_station_status(
    demo_stations, demo_incidents, demo_trust, as_of_date="2026-01-08"
)
display(demo[["location_id", "location_name", "status", "trust_score", "incident_rule_ids"]])

fig, ax = plt.subplots(figsize=(6, 5))
for name, group in demo.groupby("status"):
    ax.scatter(group["lon"], group["lat"], s=140, c=HEX[name], label=name)
    for _, row in group.iterrows():
        ax.annotate(row["location_name"].split()[0], (row["lon"], row["lat"]), fontsize=8)
ax.legend()
ax.set_title("Synthetic map: all four statuses")
ax.set_xlabel("lon")
ax.set_ylabel("lat")
plt.tight_layout()
plt.show()

## Takeaways

- Coordinates come from **bronze/conformed**, not gold
- Map status is a **join**, not a new model
- Empty fusion gold still shows **monitored** + **quality_only**
- Interactive map: `streamlit run dashboard/app.py` → Station map tab